In [169]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import gpytorch
from gpytorch.models import ExactGP
from gpytorch.means import ConstantMean
from gpytorch.kernels import ScaleKernel, RBFKernel, MaternKernel, AdditiveKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood

In [150]:
device = torch.device("mps")

In [140]:
# --- 1. Загрузка данных ---
df = pd.read_csv("ourall.csv")
df = df[df['B4TOD4'] != 0.005]
df = df[df['OMEGA5'] != 15]
df = df[df['OMEGA5'] != -15]
PARAMS = [col for col in df.columns if col != "PT_LOSS"]
X = df[PARAMS].values
y = df["PT_LOSS"].values #.reshape(-1, 1)

# --- 2. min/max по параметрам ---
param_bounds = {col: (df[col].min(), df[col].max()) for col in PARAMS}

# --- 3. Нормализация данных (очень желательно для GPs) ---
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
y_scaler = StandardScaler()
# y_scaled = y_scaler.fit_transform(y).flatten()
# --- 4. Делим на train и pool ---
np.random.seed(42)
INIT_SIZE = 10
initial_idx = np.random.choice(len(X), size=INIT_SIZE, replace=False)
X_train, y_train = X_scaled[initial_idx], y[initial_idx]
X_pool, y_pool = (
    np.delete(X_scaled, initial_idx, axis=0),
    np.delete(y, initial_idx, axis=0),
)

In [170]:
class GPRegressionModel(ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = ConstantMean()
        self.covar_module = (
            ScaleKernel(MaternKernel(nu=1.5, ard_num_dims=train_x.shape[1])) +
            ScaleKernel(RBFKernel(ard_num_dims=train_x.shape[1]))
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

In [171]:
# --- 6. Обучение GP ---
def train_gp(model, likelihood, train_x, train_y, num_iter=750, lr=0.1):
    model.set_train_data(train_x, train_y, strict=False)
    model.train()
    likelihood.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mll = ExactMarginalLogLikelihood(likelihood, model)
    for i in range(num_iter):
        optimizer.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        if torch.isnan(loss):
            print("NaN loss detected, breaking")
            break
        loss.backward()
        optimizer.step()

In [172]:
def select_next_idx_uncertainty(model, X_pool):
    model.eval()
    X_tensor = torch.tensor(X_pool, dtype=torch.float32).to(device)
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        preds = model(X_tensor)
        stds = preds.variance.sqrt().cpu().numpy().flatten()
    idx = np.argmax(stds)
    return idx

In [173]:
# --- 7. "Активная" стратегия выбора: шаг по sign(grad) ---
def select_next_idx(model, X_train, X_pool, y_train):
    # 1. Вычисляем градиент по train (на последней добавленной точке)
    model.eval()
    x_last = torch.tensor([X_train[-1]], dtype=torch.float32, requires_grad=True)
    y_pred = model(x_last)
    y_pred.variance.backward()
    grad = x_last.grad.detach().numpy().flatten()
    sign_grad = np.sign(grad)
    # 2. Шагаем от последней точки по sign(grad) (шаг 1% диапазона)
    step = 0.01
    x_next = X_train[-1] + step * sign_grad
    # Ограничиваем min/max
    x_next = np.clip(x_next, 0.0, 1.0)  # после скейлинга 0-1
    # 3. Ищем ближайшую точку в пуле (по L2)
    dists = np.linalg.norm(X_pool - x_next, axis=1)
    idx = np.argmin(dists)
    return idx

In [178]:
train_x = torch.tensor(X_train, dtype=torch.float32).to(device)
train_y = torch.tensor(y_train, dtype=torch.float32).to(device)
likelihood = GaussianLikelihood().to(device)
model = GPRegressionModel(train_x, train_y, likelihood).to(device)

In [179]:
N_QUERIES = 30
for it in range(N_QUERIES):
    # Дообучаем модель на текущих данных
    train_x = torch.tensor(X_train, dtype=torch.float32).to(device)
    train_y = torch.tensor(y_train, dtype=torch.float32).to(device)
    train_gp(model, likelihood, train_x, train_y, num_iter=500, lr=0.05)

    idx = select_next_idx_uncertainty(model, X_pool)
    # Добавляем точку с макс. std
    X_train = np.vstack([X_train, X_pool[idx]])
    y_train = np.append(y_train, y_pool[idx])
    X_pool = np.delete(X_pool, idx, axis=0)
    y_pool = np.delete(y_pool, idx, axis=0)
    print(f"Step {it+1}: added point {idx}, pool left: {len(X_pool)}")
    model.eval()
    likelihood.eval()
    X_hold = torch.tensor(X_pool, dtype=torch.float32).to(device)
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        preds = model(X_hold)
        y_pred = preds.mean.cpu().numpy().flatten()
        # Инвертируем стандартизацию
        y_true = y_pool
        # y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()
        # y_true = y_scaler.inverse_transform(y_pool.reshape(-1, 1)).flatten()

    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"Final MAE: {mae:.4f}, R2: {r2:.4f}")

Step 1: added point 3432, pool left: 3830
Step 2: added point 3607, pool left: 3829
Step 3: added point 640, pool left: 3828
Step 4: added point 933, pool left: 3827
Step 5: added point 917, pool left: 3826
Step 6: added point 3496, pool left: 3825
Step 7: added point 341, pool left: 3824
Step 8: added point 3528, pool left: 3823
Step 9: added point 3279, pool left: 3822
Step 10: added point 3721, pool left: 3821
Step 11: added point 513, pool left: 3820
Step 12: added point 3687, pool left: 3819
Step 13: added point 3630, pool left: 3818
Step 14: added point 505, pool left: 3817
Step 15: added point 598, pool left: 3816
Step 16: added point 3483, pool left: 3815
Step 17: added point 3205, pool left: 3814
Step 18: added point 3377, pool left: 3813
Step 19: added point 533, pool left: 3812
Step 20: added point 818, pool left: 3811
Step 21: added point 533, pool left: 3810
Step 22: added point 3583, pool left: 3809
Step 23: added point 819, pool left: 3808
Step 24: added point 1587, pool

In [180]:
# train_gp(
#     model,
#     likelihood,
#     torch.tensor(X_train, dtype=torch.float32).to(device),
#     torch.tensor(y_train, dtype=torch.float32).to(device),
#     num_iter=1000,
#     lr=0.02,
# )
model.eval()
likelihood.eval()
X_hold = torch.tensor(X_pool, dtype=torch.float32).to(device)
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    preds = model(X_hold)
    y_pred = preds.mean.cpu().numpy().flatten()
    # Инвертируем стандартизацию
    y_true = y_pool
    # y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()
    # y_true = y_scaler.inverse_transform(y_pool.reshape(-1, 1)).flatten()

mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print(f"Final MAE: {mae:.4f}, R2: {r2:.4f}")

Final MAE: 0.0208, R2: 0.8963


In [168]:
y_true.min(), y_true.max()

(np.float64(-0.107619), np.float64(0.725187))

In [167]:
y_pred.min(), y_pred.max()

(np.float32(-0.019192815), np.float32(0.5859593))

In [136]:
# --- 8. Активное обучение ---
N_QUERIES = 70
model, likelihood = None, None
for it in range(N_QUERIES):
    # 1. Обучаем GP
    model, likelihood = train_gp(
        X_train, y_train, num_iter=10
    )
    # 2. Выбор новой точки
    idx = select_next_idx(model, X_train, X_pool, y_train)
    # 3. Добавляем в train
    X_train = np.vstack([X_train, X_pool[idx]])
    y_train = np.append(y_train, y_pool[idx])
    # 4. Удаляем из пула
    X_pool = np.delete(X_pool, idx, axis=0)
    y_pool = np.delete(y_pool, idx, axis=0)
    print(f"Step {it + 1}: added point {idx}, pool left: {len(X_pool)}")

Step 1: added point 1039, pool left: 3737
Step 2: added point 1037, pool left: 3736
Step 3: added point 1655, pool left: 3735
Step 4: added point 1656, pool left: 3734
Step 5: added point 1657, pool left: 3733
Step 6: added point 1658, pool left: 3732
Step 7: added point 1659, pool left: 3731
Step 8: added point 1660, pool left: 3730
Step 9: added point 2591, pool left: 3729
Step 10: added point 2589, pool left: 3728
Step 11: added point 2587, pool left: 3727
Step 12: added point 2585, pool left: 3726
Step 13: added point 2583, pool left: 3725
Step 14: added point 2581, pool left: 3724
Step 15: added point 3510, pool left: 3723
Step 16: added point 3511, pool left: 3722
Step 17: added point 3512, pool left: 3721
Step 18: added point 3513, pool left: 3720
Step 19: added point 3514, pool left: 3719
Step 20: added point 3515, pool left: 3718
Step 21: added point 3413, pool left: 3717
Step 22: added point 3411, pool left: 3716
Step 23: added point 3409, pool left: 3715
Step 24: added point

In [137]:
# --- 9. Финальная оценка на holdout (остатке пула) ---
model, likelihood = train_gp(X_train, y_train)
model.eval()
likelihood.eval()
with torch.no_grad():
    X_hold = torch.tensor(X_pool, dtype=torch.float32)
    y_pred = model(X_hold).mean.numpy()
mae = mean_absolute_error(y_pool, y_pred)
r2 = r2_score(y_pool, y_pred)
print(f"Final MAE: {mae:.4f}, R2: {r2:.4f}")

Final MAE: 0.0354, R2: 0.7893


In [138]:
y_pred

array([0.07865342, 0.04117784, 0.03006944, ..., 0.17612657, 0.18442735,
       0.16687974], shape=(3668,), dtype=float32)

In [139]:
y_pool

array([0.108975 , 0.061027 , 0.0462413, ..., 0.0992762, 0.116008 ,
       0.0890589], shape=(3668,))